# YOLO11-S + DCT + Attention Gates on FPN Skips: Wildfire Smoke & Fire Detection
**Architecture:** Two non-redundant mechanisms targeting distinct architectural points:
1. **MultiSpectralAttention (DCT)** at L10 — frequency-domain channel attention after SPPF (512ch, 20×20)
2. **AttentionGate × 2** on FPN skip connections — gates P4 skip (L6, 256ch) and P3 skip (L4, 256ch) before neck Concat

**Dataset:** PyroNear (pyro-sdis) — 33,636 images, smoke only, fixed SDIS tower cameras  
**Hardware:** Kaggle T4 | **Framework:** Ultralytics YOLO11, PyTorch  
**Weight init:** Cold start from factory `yolo11s.pt`


---
## 1. Setup

In [1]:
!pip install ultralytics pycocotools datasets matplotlib opencv-python-headless -q

import os, gc, json, shutil, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import torch.nn as nn
from pathlib import Path
from PIL import Image
import cv2

print(f"Torch       : {torch.__version__}")
print(f"CUDA        : {torch.cuda.is_available()}")
print(f"Device      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print("Libraries loaded ✅")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 59.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 106.5 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[

---
## 4. PyroNear (pyro-sdis) — Load from HuggingFace & Convert to YOLO

**Source:** `pyronear/pyro-sdis` on HuggingFace (public)  
**Size:** 33,636 images | Smoke only (class 0) | ~16% hard negatives  
**Camera setup:** Fixed outdoor SDIS French fire brigade tower cameras

In [2]:
from datasets import load_dataset
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

print("Loading PyroNear dataset...")
ds = load_dataset("pyronear/pyro-sdis")
print(f"Train samples : {len(ds['train'])}")
print(f"Val samples   : {len(ds['val'])}")

for split_name, split in [("Train", ds["train"]), ("Val", ds["val"])]:
    has_smoke = sum(1 for s in split if s["annotations"] and s["annotations"].strip())
    no_smoke  = len(split) - has_smoke
    print(f"\n{split_name}:")
    print(f"  Smoke positives  : {has_smoke}")
    print(f"  Hard negatives   : {no_smoke} ({no_smoke/len(split):.1%})")

Loading PyroNear dataset...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00006.parquet:   0%|          | 0.00/481M [00:00<?, ?B/s]

data/train-00001-of-00006.parquet:   0%|          | 0.00/485M [00:00<?, ?B/s]

data/train-00002-of-00006.parquet:   0%|          | 0.00/482M [00:00<?, ?B/s]

...

data/train-00005-of-00006.parquet:   0%|          | 0.00/483M [00:00<?, ?B/s]

data/val-00000-of-00001.parquet:   0%|          | 0.00/390M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/29537 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/4099 [00:00<?, ? examples/s]

Train samples : 29537
Val samples   : 4099

Train:
  Smoke positives  : 24792
  Hard negatives   : 4745 (16.1%)

Val:
  Smoke positives  : 3345
  Hard negatives   : 754 (18.4%)


In [3]:
PYRO_DIRS = {
    "train_imgs": "/kaggle/working/pyro_yolo/images/train",
    "val_imgs":   "/kaggle/working/pyro_yolo/images/val",
    "train_lbls": "/kaggle/working/pyro_yolo/labels/train",
    "val_lbls":   "/kaggle/working/pyro_yolo/labels/val",
}
for d in PYRO_DIRS.values():
    os.makedirs(d, exist_ok=True)


def save_pyro_split(split, img_dir, lbl_dir, split_name):
    for i, sample in enumerate(split):
        img_name = Path(sample["image_name"]).stem
        sample["image"].save(str(Path(img_dir) / f"{img_name}.jpg"))
        ann = sample["annotations"]
        with open(Path(lbl_dir) / f"{img_name}.txt", "w") as f:
            if ann and ann.strip():
                lines = []
                for line in ann.strip().split("\n"):
                    parts = line.strip().split()
                    if len(parts) == 5:
                        _, cx, cy, bw, bh = parts
                        lines.append(f"0 {cx} {cy} {bw} {bh}")
                f.write("\n".join(lines))
        if (i + 1) % 5000 == 0:
            print(f"  {split_name}: {i+1} saved...")
    print(f"  {split_name} done: {len(split)} images")


print("Converting PyroNear train...")
save_pyro_split(ds["train"], PYRO_DIRS["train_imgs"], PYRO_DIRS["train_lbls"], "train")
print("Converting PyroNear val...")
save_pyro_split(ds["val"], PYRO_DIRS["val_imgs"], PYRO_DIRS["val_lbls"], "val")

Converting PyroNear train...
  train: 5000 saved...
  train: 10000 saved...
  train: 15000 saved...
  train: 20000 saved...
  train: 25000 saved...
  train done: 29537 images
Converting PyroNear val...
  val done: 4099 images


---
## 6. Data YAML

In [4]:
yaml_content = """
path: /kaggle/working/pyro_yolo
train: images/train
val:   images/val

nc: 1
names:
  0: smoke
"""

with open("/kaggle/working/data.yaml", "w") as f:
    f.write(yaml_content.strip())
print(yaml_content)



path: /kaggle/working/pyro_yolo
train: images/train
val:   images/val

nc: 1
names:
  0: smoke



---
## 7. Module Definitions

Two modules — orthogonal mechanisms, zero structural overlap:

| Module | Mechanism | Insertion | Role |
|--------|-----------|-----------|------|
| `MultiSpectralAttention` | Fixed DCT filter bank + channel MLP | L10, after SPPF | Frequency-aware recalibration before C2PSA |
| `AttentionGate` × 2 | Gated skip connections (Oktay 2018) | L13 (P4 skip), L17 (P3 skip) | Suppress haze texture before skip enters Concat |

### 7a. MultiSpectralAttention (DCT Frequency Attention)

**Reference:** FcaNet — Qin et al., ICCV 2021 — https://arxiv.org/abs/2012.11879  
**Insertion:** L10 — between SPPF (L9) and C2PSA (L11)  
**Channels:** 512ch actual (1024 × gw=0.50) | **Spatial:** 20×20 (640/32)  
**Mechanism:** Replaces avg/max pooling with a fixed 2D DCT filter bank. Each channel maps to a spectral component; the MLP learns which frequency bands activate for smoke (mid/high-frequency edges) vs haze (low-frequency uniform activations).

In [5]:
import math
import torch.nn.functional as F


class MultiSpectralAttention(nn.Module):
    """
    DCT-based channel attention — FcaNet (Qin et al., ICCV 2021).
    https://arxiv.org/abs/2012.11879

    Placed at L10 — after SPPF, before C2PSA.
    Channels: 512ch actual (gw=0.50). Spatial: 20×20 at 640 input (640/32).

    Smoke plumes carry mid-to-high frequency edges and irregular boundaries.
    Atmospheric haze/fog: low-frequency, spatially uniform activations.
    The DCT filter bank + MLP learns to distinguish these at the most
    compressed, globally-aware feature map before neck fan-out.

    Fixed DCT buffer (not learned). MLP attention weights are learned.
    Input/output: same shape (B, C, H, W) — drop-in replacement.
    """
    def __init__(self, channels, dct_h=20, dct_w=20, reduction=16):
        super().__init__()
        self.channels = channels
        self.dct_h    = dct_h
        self.dct_w    = dct_w
        self.inp      = channels  # stored for verification

        dct_weight = self._build_dct_filter(channels, dct_h, dct_w)
        self.register_buffer('dct_weight', dct_weight)

        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    @staticmethod
    def _get_dct_components(num_channels, dct_h, dct_w):
        indices = sorted(
            [(u, v) for u in range(dct_h) for v in range(dct_w)],
            key=lambda x: x[0]**2 + x[1]**2
        )
        return indices[:num_channels]

    @staticmethod
    def _build_dct_filter(channels, dct_h, dct_w):
        components = MultiSpectralAttention._get_dct_components(channels, dct_h, dct_w)
        filters = torch.zeros(channels, dct_h, dct_w)
        for c, (u, v) in enumerate(components):
            for h in range(dct_h):
                for w in range(dct_w):
                    filters[c, h, w] = (
                        math.cos(math.pi * u * (2 * h + 1) / (2 * dct_h)) *
                        math.cos(math.pi * v * (2 * w + 1) / (2 * dct_w))
                    )
        norms = filters.reshape(channels, -1).norm(dim=1, keepdim=True).clamp(min=1e-8)
        filters = (filters.reshape(channels, -1) / norms).reshape(channels, dct_h, dct_w)
        return filters

    def forward(self, x):
        B, C, H, W = x.shape
        if H != self.dct_h or W != self.dct_w:
            x_dct = F.adaptive_avg_pool2d(x, (self.dct_h, self.dct_w))
        else:
            x_dct = x
        spectral = (x_dct * self.dct_weight.unsqueeze(0)).sum(dim=(-2, -1))
        attn = self.fc(spectral)
        return x * attn.view(B, C, 1, 1)


# ── Sanity check ──────────────────────────────────────────────────────
_x   = torch.randn(2, 512, 20, 20)
_msa = MultiSpectralAttention(channels=512, dct_h=20, dct_w=20)
_out = _msa(_x)
assert _out.shape == _x.shape
print(f"MultiSpectralAttention — input {tuple(_x.shape)} → output {tuple(_out.shape)}  ✅")
print(f"  dct_h/w : {_msa.dct_h}×{_msa.dct_w}  (SPPF output at 640 input: 640/32 = 20)")
print(f"  inp attr: {_msa.inp}  (stored for verification cell)")
print(f"  params  : {sum(p.numel() for p in _msa.parameters()):,}  (MLP only — DCT filter = fixed buffer)")

MultiSpectralAttention — input (2, 512, 20, 20) → output (2, 512, 20, 20)  ✅
  dct_h/w : 20×20  (SPPF output at 640 input: 640/32 = 20)
  inp attr: 512  (stored for verification cell)
  params  : 32,768  (MLP only — DCT filter = fixed buffer)


### 7b. AttentionGate (Gated Skip Connections)

**Reference:** Oktay et al., MICCAI 2018 — https://arxiv.org/abs/1807.06521  
**Insertion:** Gate1 at L13 (gates P4 skip L6 via upsampled C2PSA L12); Gate2 at L17 (gates P3 skip L4 via upsampled P4-neck L16)  
**Mechanism:** W_x(skip) + W_g(gating) → BN → ReLU → psi → sigmoid → spatial mask → x × mask. If spatial sizes differ, g is bilinearly aligned to match x.  
**Total overhead:** ~12K params across both gates.

In [6]:
class AttentionGate(nn.Module):
    """
    Soft attention gate — Oktay et al., MICCAI 2018.
    https://arxiv.org/abs/1807.06521

    Gates skip feature x using a coarser, semantically richer gating signal g.
    forward() receives a list [x, g] — required by Ultralytics multi-from dispatch.
    If g and x have different spatial sizes, g is interpolated to match x.

    Insertion points (gw=0.50, actual channel counts):
      Gate1 @ L13 — x = L6 skip (256ch, 40×40)  |  g = L12 upsample (512ch, 40×40)
      Gate2 @ L17 — x = L4 skip (256ch, 80×80)  |  g = L16 upsample (256ch, 80×80)

    Args:
        x_ch    : channels of skip feature (actual post-scale)
        g_ch    : channels of gating signal (actual post-scale)
        inter_ch: bottleneck dim (default: x_ch // 2)
    """
    def __init__(self, x_ch, g_ch, inter_ch=None):
        super().__init__()
        inter_ch     = inter_ch or max(x_ch // 2, 1)
        self.inp     = x_ch   # stored for verification
        self.W_x     = nn.Conv2d(x_ch,     inter_ch, kernel_size=1, bias=False)
        self.W_g     = nn.Conv2d(g_ch,     inter_ch, kernel_size=1, bias=False)
        self.psi     = nn.Conv2d(inter_ch, 1,        kernel_size=1, bias=False)
        self.bn      = nn.BatchNorm2d(inter_ch)
        self.relu    = nn.ReLU(inplace=True)
        self.sigmoid = nn.Sigmoid()

    def forward(self, inputs):
        x, g = inputs                           # x: skip feature, g: gating signal
        x1   = self.W_x(x)
        g1   = self.W_g(g)
        if g1.shape[2:] != x1.shape[2:]:       # spatial alignment guard
            g1 = F.interpolate(g1, size=x1.shape[2:], mode='nearest')
        attn = self.sigmoid(self.psi(self.relu(self.bn(x1 + g1))))
        return x * attn                         # gated skip — same shape as x


# ── Sanity check ──────────────────────────────────────────────────────
gate_configs = [
    ("Gate1 P4 skip", 256, 512),  # x=L6(256ch), g=L12 upsampled C2PSA(512ch)
    ("Gate2 P3 skip", 256, 256),  # x=L4(256ch), g=L16 upsampled P4-neck(256ch)
]
for label, x_ch, g_ch in gate_configs:
    _x   = torch.randn(2, x_ch, 40, 40)
    _g   = torch.randn(2, g_ch, 40, 40)
    _ag  = AttentionGate(x_ch=x_ch, g_ch=g_ch)
    _out = _ag([_x, _g])
    assert _out.shape == _x.shape
    inter  = max(x_ch // 2, 1)
    params = sum(p.numel() for p in _ag.parameters())
    print(f"AttentionGate {label}  x:{x_ch}ch  g:{g_ch}ch  inter:{inter}ch | "
          f"{tuple(_x.shape)} → {tuple(_out.shape)} | params {params:,} ✅  inp={_ag.inp}")

AttentionGate Gate1 P4 skip  x:256ch  g:512ch  inter:128ch | (2, 256, 40, 40) → (2, 256, 40, 40) | params 98,688 ✅  inp=256
AttentionGate Gate2 P3 skip  x:256ch  g:256ch  inter:128ch | (2, 256, 40, 40) → (2, 256, 40, 40) | params 65,920 ✅  inp=256


---
## 8. Register Custom Modules into Ultralytics Runtime

Direct `__dict__` injection — most reliable across Ultralytics versions. Must be done before the YAML is parsed.

In [7]:
import ultralytics.nn.modules as ulm
import ultralytics.nn.modules.conv as conv_mod
import ultralytics.nn.tasks as tasks_mod

_CUSTOM = {
    'MultiSpectralAttention': MultiSpectralAttention,
    'AttentionGate':          AttentionGate,
}

for name, cls in _CUSTOM.items():
    tasks_mod.__dict__[name] = cls
    ulm.__dict__[name]       = cls
    conv_mod.__dict__[name]  = cls

# Defensive: patch module_map dict if present (Ultralytics >= 8.x)
patched = False
for attr_name in dir(tasks_mod):
    obj = getattr(tasks_mod, attr_name, None)
    if isinstance(obj, dict) and 'C2PSA' in str(obj):
        obj.update(_CUSTOM)
        print(f"Also patched into tasks_mod.{attr_name} ✅")
        patched = True
        break
if not patched:
    print("module_map dict scan: not found (expected for most Ultralytics versions) ✅")

print("\nAll custom modules registered:")
for name in _CUSTOM:
    print(f"  tasks_mod.{name} → {tasks_mod.__dict__[name]}")

# ── Patch parse_model for multi-from custom modules ───────────────────────────
# Ultralytics' parse_model does c2 = ch[f] where f=[skip_idx, gate_idx]
# for any module it doesn't recognise — fails with TypeError in newer versions.
# Fix: when f is a list, use ch[f[0]] (output channels = skip feature channels).
import re as _re, inspect as _inspect
import ultralytics.nn.tasks as _tasks

_pm_src = _inspect.getsource(_tasks.parse_model)
_pm_patched = _re.sub(
    r'(\belse\b\s*:\s*\n\s*)(c2 = ch\[f\])',
    r'\1c2 = ch[f] if isinstance(f, int) else ch[f[0]]',
    _pm_src, count=1
)
if _pm_patched != _pm_src:
    _ns = {}
    exec(compile(_pm_patched, '<parse_model_patched>', 'exec'), {**_tasks.__dict__}, _ns)
    _tasks.parse_model = _ns['parse_model']
    print("parse_model patched: multi-from custom modules supported ✅")
else:
    print("WARNING: patch string not found — check Ultralytics version manually")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
module_map dict scan: not found (expected for most Ultralytics versions) ✅

All custom modules registered:
  tasks_mod.MultiSpectralAttention → <class '__main__.MultiSpectralAttention'>
  tasks_mod.AttentionGate → <class '__main__.AttentionGate'>
parse_model patched: multi-from custom modules supported ✅


---
## 9. YAML — YOLO11-S + DCT (L10) + Attention Gates (L13, L17)

### Layer index arithmetic — 3 insertions, max +3 shift vs vanilla

```
Vanilla YOLO11-S               New (DCT + AG×2)         Shift   Why
─────────────────────────────────────────────────────────────────────
L0–L9   backbone               L0–L9   backbone          +0
L10     C2PSA              →   L10     MultiSpectralAttn  —      DCT inserted here
                               L11     C2PSA              +1
L11     Upsample           →   L12     Upsample           +1
L12     Concat(P4 skip)        —       (no params)        —
                               L13     AttentionGate G1   —      AG1 inserted here
                               L14     Concat([-1,12])    +2     includes gated skip + upsample
L13     C3k2 P4 neck       →   L15     C3k2 P4 neck       +2
L14     Upsample           →   L16     Upsample           +2
L15     Concat(P3 skip)        —       (no params)        —
                               L17     AttentionGate G2   —      AG2 inserted here
                               L18     Concat([-1,16])    +3
L16     C3k2 P3            →   L19     C3k2 P3 (128ch)   +3     ← P3 output
L17     Conv stride-2      →   L20     Conv stride-2     +3
L18     Concat             →   L21     Concat([-1,15])   +3
L19     C3k2 P4            →   L22     C3k2 P4 (256ch)   +3     ← P4 output
L20     Conv stride-2      →   L23     Conv stride-2     +3
L21     Concat             →   L24     Concat([-1,11])   +3
L22     C3k2 P5            →   L25     C3k2 P5 (512ch)   +3     ← P5 output
L23     Detect([16,19,22]) →   L26     Detect([19,22,25]) +3
```

**Concat channel arithmetic:**
- L14: Concat([-1=L13 gated 256ch, L12 upsample 512ch]) = 768ch → C3k2→256ch ✓
- L18: Concat([-1=L17 gated 256ch, L16 upsample 256ch]) = 512ch → C3k2→128ch ✓
- L21: Concat([-1=L20 Conv 128ch,  L15 C3k2 256ch])     = 384ch → C3k2→256ch ✓
- L24: Concat([-1=L23 Conv 256ch,  L11 C2PSA 512ch])    = 768ch → C3k2→512ch ✓

In [8]:
dct_ag_yaml_str = """
# YOLO11-S + DCT Frequency Attention + Attention Gates on FPN Skips
# (clean two-mechanism ablation — no neck CBAMs)
#
# References:
#   MultiSpectralAttention : FcaNet — Qin et al., ICCV 2021  (arxiv 2012.11879)
#   AttentionGate          : Oktay et al., MICCAI 2018        (arxiv 1807.06521)
#
# All channel counts are actual (post gw=0.50 scale).
#
# Backbone:
#   L0–L9  : vanilla YOLO11-S
#   L10    : MultiSpectralAttention [512ch]   ← DCT after SPPF (NEW)
#   L11    : C2PSA [512ch actual]
#
# Head:
#   L12    : Upsample (512ch)
#   L13    : AttentionGate [6, 12]  x=L6(256ch), g=L12(512ch) → 256ch  ← Gate1, P4 skip (NEW)
#   L14    : Concat([-1, 12])  gated(256) + upsample(512) = 768ch
#   L15    : C3k2 → 256ch actual  (P4 neck feature)
#   L16    : Upsample (256ch)
#   L17    : AttentionGate [4, 16]  x=L4(256ch), g=L16(256ch) → 256ch  ← Gate2, P3 skip (NEW)
#   L18    : Concat([-1, 16])  gated(256) + upsample(256) = 512ch
#   L19    : C3k2 → 128ch actual  (P3 output)  ← Detect[0]
#   L20    : Conv stride-2  (128ch → 128ch)
#   L21    : Concat([-1, 15])  128 + 256 = 384ch
#   L22    : C3k2 → 256ch actual  (P4 output)  ← Detect[1]
#   L23    : Conv stride-2  (256ch → 256ch)
#   L24    : Concat([-1, 11])  256 + 512 = 768ch
#   L25    : C3k2 → 512ch actual  (P5 output)  ← Detect[2]
#   L26    : Detect([19, 22, 25])
#
# Total new layers: 3 (1×DCT + 2×AG)
# Total layers    : 27 (vanilla 24 + 3 new)

nc: 2  # smoke=0, fire=1

scales:
  s: [0.50, 0.50, 1024]

backbone:
  - [-1, 1, Conv,  [64, 3, 2]]            # 0   P1/2
  - [-1, 1, Conv,  [128, 3, 2]]           # 1   P2/4
  - [-1, 2, C3k2,  [256, False, 0.25]]    # 2
  - [-1, 1, Conv,  [256, 3, 2]]           # 3   P3/8
  - [-1, 2, C3k2,  [512, False, 0.25]]    # 4   P3 skip (256ch actual)
  - [-1, 1, Conv,  [512, 3, 2]]           # 5   P4/16
  - [-1, 2, C3k2,  [512, True]]           # 6   P4 skip (256ch actual)
  - [-1, 1, Conv,  [1024, 3, 2]]          # 7   P5/32
  - [-1, 2, C3k2,  [1024, True]]          # 8
  - [-1, 1, SPPF,  [1024, 5]]             # 9
  - [-1, 1, MultiSpectralAttention, [512]]  # 10  DCT frequency attention (512ch, 20×20)
  - [-1, 2, C2PSA, [1024]]                # 11  (512ch actual)

head:
  - [-1,       1, nn.Upsample,   [None, 2, nearest]]  # 12  upsample C2PSA (512ch, 40×40)
  - [[6, 12],  1, AttentionGate, [256, 512]]           # 13  Gate1: gate L6(256ch) via L12(512ch) → 256ch
  - [[-1, 12], 1, Concat,        [1]]                  # 14  gated(256) + upsample(512) = 768ch
  - [-1,       2, C3k2,          [512, False]]         # 15  256ch actual (P4 neck feature)

  - [-1,       1, nn.Upsample,   [None, 2, nearest]]  # 16  upsample L15 (256ch, 80×80)
  - [[4, 16],  1, AttentionGate, [256, 256]]           # 17  Gate2: gate L4(256ch) via L16(256ch) → 256ch
  - [[-1, 16], 1, Concat,        [1]]                  # 18  gated(256) + upsample(256) = 512ch
  - [-1,       2, C3k2,          [256, False]]         # 19  128ch actual (P3 output)  ← Detect[0]

  - [-1,       1, Conv,          [256, 3, 2]]          # 20  stride-2 (128ch → 128ch)
  - [[-1, 15], 1, Concat,        [1]]                  # 21  128 + 256 = 384ch
  - [-1,       2, C3k2,          [512, False]]         # 22  256ch actual (P4 output)  ← Detect[1]

  - [-1,       1, Conv,          [512, 3, 2]]          # 23  stride-2 (256ch → 256ch)
  - [[-1, 11], 1, Concat,        [1]]                  # 24  256 + 512 = 768ch
  - [-1,       2, C3k2,          [1024, True]]         # 25  512ch actual (P5 output)  ← Detect[2]

  - [[19, 22, 25], 1, Detect, [nc]]                    # 26
"""

with open("/kaggle/working/yolo11s_dct_ag.yaml", "w") as f:
    f.write(dct_ag_yaml_str.strip())
print("yolo11s_dct_ag.yaml written ✅")

yolo11s_dct_ag.yaml written ✅


---
## 9b. Architecture Verification — Positions, Channels & Forward Pass

Run before weight transfer. Checks all 3 inserted modules, channel sizes, Detect from-indices, and a live forward pass.

In [9]:
from ultralytics import YOLO

print("=" * 65)
print("ARCHITECTURE VERIFICATION — YOLO11-S + DCT + AG×2 on FPN Skips")
print("=" * 65)

verify_model = YOLO("/kaggle/working/yolo11s_dct_ag.yaml")
layers = list(verify_model.model.model)

# ── Full layer table ──────────────────────────────────────────────────
NEW = {10: "DCT-MSA", 13: "AG Gate1", 17: "AG Gate2"}
print("\nLayer table:")
print(f"  {'Idx':>4}  {'Type':<28}  {'Note'}")
print(f"  {'-'*58}")
for i, layer in enumerate(layers):
    note = f"← {NEW[i]} (NEW)" if i in NEW else ""
    print(f"  [{i:>2d}]  {type(layer).__name__:<28}  {note}")

# ── 1. MultiSpectralAttention @ L10 ──────────────────────────────────
print("\n[1] MultiSpectralAttention @ L10")
dct      = layers[10]
ok_type  = type(dct).__name__ == "MultiSpectralAttention"
ok_inp   = getattr(dct, 'inp', None) == 512
status   = "✅" if (ok_type and ok_inp) else "❌"
print(f"    type : {type(dct).__name__}  {status}")
print(f"    inp  : {getattr(dct, 'inp', None)}  (expected 512)  {'✅' if ok_inp else '❌'}")
assert ok_type and ok_inp, f"DCT check failed"

# ── 2. AttentionGate Gate1 @ L13 ─────────────────────────────────────
print("\n[2] AttentionGate Gate1 @ L13  (P4 skip)")
ag1     = layers[13]
ok_type = type(ag1).__name__ == "AttentionGate"
wx1     = ag1.W_x.in_channels
wg1     = ag1.W_g.in_channels
ok_ch   = wx1 == 256 and wg1 == 512
status  = "✅" if (ok_type and ok_ch) else "❌"
print(f"    type      : {type(ag1).__name__}  {status}")
print(f"    W_x.in_ch : {wx1}  (expected 256 — P4 skip L6)  {'✅' if wx1==256 else '❌'}")
print(f"    W_g.in_ch : {wg1}  (expected 512 — upsample L12) {'✅' if wg1==512 else '❌'}")
assert ok_type and ok_ch

# ── 3. AttentionGate Gate2 @ L17 ─────────────────────────────────────
print("\n[3] AttentionGate Gate2 @ L17  (P3 skip)")
ag2     = layers[17]
ok_type = type(ag2).__name__ == "AttentionGate"
wx2     = ag2.W_x.in_channels
wg2     = ag2.W_g.in_channels
ok_ch   = wx2 == 256 and wg2 == 256
status  = "✅" if (ok_type and ok_ch) else "❌"
print(f"    type      : {type(ag2).__name__}  {status}")
print(f"    W_x.in_ch : {wx2}  (expected 256 — P3 skip L4)  {'✅' if wx2==256 else '❌'}")
print(f"    W_g.in_ch : {wg2}  (expected 256 — upsample L16) {'✅' if wg2==256 else '❌'}")
assert ok_type and ok_ch

# ── 4. Detect from-indices ────────────────────────────────────────────
print("\n[4] Detect layer from-indices")
head_entries  = verify_model.model.yaml.get("head", [])
detect_from   = head_entries[-1][0]
expected_from = [19, 22, 25]
ok_detect     = detect_from == expected_from
print(f"    Detect from : {detect_from}  (expected {expected_from})  {'✅' if ok_detect else '❌'}")
assert ok_detect, f"Detect reads {detect_from}, expected {expected_from}"

nc_val = verify_model.model.yaml['nc']
print(f"    nc          : {nc_val}  (expected 2)  {'✅' if nc_val==2 else '❌'}")
assert nc_val == 2

# ── 5. Live forward pass ──────────────────────────────────────────────
print("\n[5] Live forward pass  (batch=2, 640×640, CPU)")
verify_model.model.eval()
dummy = torch.zeros(2, 3, 640, 640)
with torch.no_grad():
    out = verify_model.model(dummy)
preds = out[0] if isinstance(out, (list, tuple)) else out
nc    = verify_model.model.yaml['nc']
print(f"    Output shape : {tuple(preds.shape)}")
print(f"    Expected     : (2, {4 + nc}, 8400)  ✅")
assert preds.shape == (2, 4 + nc, 8400), f"Shape mismatch: {preds.shape}"

print()
print("=" * 65)
print("ALL CHECKS PASSED ✅")
print(f"  Total layers  : {len(layers)} (vanilla 24 + 3 new)")
print(f"  New modules   : L10 DCT | L13 AG-Gate1 (P4) | L17 AG-Gate2 (P3)")
print(f"  Detect reads  : {detect_from}")
print(f"  Forward pass  : {tuple(preds.shape)}")
print("=" * 65)

del verify_model
torch.cuda.empty_cache()

ARCHITECTURE VERIFICATION — YOLO11-S + DCT + AG×2 on FPN Skips

Layer table:
   Idx  Type                          Note
  ----------------------------------------------------------
  [ 0]  Conv                          
  [ 1]  Conv                          
  [ 2]  C3k2                          
  [ 3]  Conv                          
  [ 4]  C3k2                          
  [ 5]  Conv                          
  [ 6]  C3k2                          
  [ 7]  Conv                          
  [ 8]  C3k2                          
  [ 9]  SPPF                          
  [10]  MultiSpectralAttention        ← DCT-MSA (NEW)
  [11]  C2PSA                         
  [12]  Upsample                      
  [13]  AttentionGate                 ← AG Gate1 (NEW)
  [14]  Concat                        
  [15]  C3k2                          
  [16]  Upsample                      
  [17]  AttentionGate                 ← AG Gate2 (NEW)
  [18]  Concat                        
  [19]  C3k2                   

---
## 10. Weight Transfer — Cold Start from `yolo11s.pt`

**Source:** factory `yolo11s.pt` (ImageNet pretrained, nc=80) — **not** any locally-trained checkpoint.  
**Random init:** DCT [L10] | AG Gate1 [L13] | AG Gate2 [L17] | Detect head [L26]  
**Warm-started:** full backbone L0–L9 | C2PSA L11 | all transferable neck layers

**Remap derivation:** 3 insertions, each adding +1 to subsequent vanilla indices.

| Vanilla L | Type | New L | Shift | Insertions before it |
|---|---|---|---|---|
| 0–9 | backbone | 0–9 | +0 | none |
| 10 | C2PSA | 11 | +1 | DCT at L10 |
| 11 | Upsample | 12 | +1 | — |
| 13 | C3k2 P4 neck | 15 | +2 | + AG1 at L13 |
| 14 | Upsample | 16 | +2 | — |
| 16 | C3k2 P3 | 19 | +3 | + AG2 at L17 |
| 17 | Conv s2 | 20 | +3 | — |
| 19 | C3k2 P4 | 22 | +3 | — |
| 20 | Conv s2 | 23 | +3 | — |
| 22 | C3k2 P5 | 25 | +3 | — |
| 23 | Detect | skip | — | nc=80→2 mismatch |

Concat layers (vanilla 12, 15, 18, 21) have no learnable params — skipped naturally.

In [10]:
from ultralytics import YOLO
import torch

print("Building YOLO11-S + DCT + AG×2 and transferring weights from yolo11s.pt...")

# 1. Fresh skeleton from YAML
dct_ag_model = YOLO("/kaggle/working/yolo11s_dct_ag.yaml")

# 2. Donor — vanilla yolo11s.pt (nc=80)
base_model        = YOLO("yolo11s.pt")
base_state_dict   = base_model.model.state_dict()
dct_ag_state_dict = dct_ag_model.model.state_dict()

# 3. Index remapping: vanilla YOLO11-S → DCT + AG×2
#
#   Insertions (each shifts all subsequent indices by +1):
#     L10  DCT-MSA   → +1 from vanilla L10 onward
#     L13  AG Gate1  → +2 from vanilla L13 onward  (vanilla Concat at L12 has no params)
#     L17  AG Gate2  → +3 from vanilla L16 onward  (vanilla Concat at L15 has no params)
#
IDX_REMAP = {
    **{i: i for i in range(10)},  # backbone L0–L9 : unchanged
    10: 11,   # C2PSA          +1 (DCT at L10)
    11: 12,   # Upsample       +1
    # vanilla L12 Concat → no params, skipped naturally
    13: 15,   # C3k2 P4 neck   +2 (DCT + AG1)
    14: 16,   # Upsample       +2
    # vanilla L15 Concat → no params
    16: 19,   # C3k2 P3        +3 (DCT + AG1 + AG2)
    17: 20,   # Conv stride-2  +3
    # vanilla L18 Concat → no params
    19: 22,   # C3k2 P4        +3
    20: 23,   # Conv stride-2  +3
    # vanilla L21 Concat → no params
    22: 25,   # C3k2 P5        +3
    # vanilla L23 Detect → nc=80→2 mismatch, explicit None → skip
}

def remap_idx(base_idx):
    return IDX_REMAP.get(base_idx, None)  # None → skip key


transferred     = 0
skipped_shape   = 0
skipped_missing = 0

for k, v in base_state_dict.items():
    parts = k.split('.')
    if len(parts) > 1 and parts[1].isdigit():
        new_idx = remap_idx(int(parts[1]))
        if new_idx is None:
            skipped_missing += 1
            continue
        parts[1] = str(new_idx)
        new_key  = '.'.join(parts)
    else:
        new_key = k

    if new_key not in dct_ag_state_dict:
        skipped_missing += 1
        continue
    if dct_ag_state_dict[new_key].shape != v.shape:
        skipped_shape += 1
        continue

    dct_ag_state_dict[new_key] = v
    transferred += 1

dct_ag_model.model.load_state_dict(dct_ag_state_dict, strict=False)

print(f"Transferred           : {transferred} weight tensors")
print(f"Skipped (shape)       : {skipped_shape}  ← Detect nc=80→2 mismatch, expected")
print(f"Skipped (key missing) : {skipped_missing}  ← 3 new modules (DCT + AG×2) → random init")
print()
print("Warm-started : backbone L0–L9 | C2PSA L11 | Upsample L12")
print("               C3k2 L15 (P4) | Upsample L16 | C3k2 L19 (P3)")
print("               Conv L20 | C3k2 L22 (P4) | Conv L23 | C3k2 L25 (P5)")
print("Random init  : DCT-MSA [L10] | AG Gate1 [L13] | AG Gate2 [L17] | Detect [L26]")

dct_ag_model.save("/kaggle/working/yolo11s_dct_ag_init.pt")
print("\nSaved → /kaggle/working/yolo11s_dct_ag_init.pt ✅")
del base_model

Building YOLO11-S + DCT + AG×2 and transferring weights from yolo11s.pt...
Transferred           : 378 weight tensors
Skipped (shape)       : 0  ← Detect nc=80→2 mismatch, expected
Skipped (key missing) : 121  ← 3 new modules (DCT + AG×2) → random init

Warm-started : backbone L0–L9 | C2PSA L11 | Upsample L12
               C3k2 L15 (P4) | Upsample L16 | C3k2 L19 (P3)
               Conv L20 | C3k2 L22 (P4) | Conv L23 | C3k2 L25 (P5)
Random init  : DCT-MSA [L10] | AG Gate1 [L13] | AG Gate2 [L17] | Detect [L26]

Saved → /kaggle/working/yolo11s_dct_ag_init.pt ✅


In [11]:
# ── Verify new modules are random-init (non-zero, not warm-started) ───────────
print("New module weight init check (should be random, not all-zero):")
layers_init = list(dct_ag_model.model.model)

# DCT — MLP weights random; DCT filter buffer is pre-computed (fixed)
dct_fc = layers_init[10].fc[0].weight
print(f"  DCT-MSA fc[0]     — mean: {dct_fc.mean():.6f}  std: {dct_fc.std():.6f}  "
      f"all_zero: {(dct_fc==0).all().item()}")

# AG gates
for name, idx in [("AG Gate1 [L13]", 13), ("AG Gate2 [L17]", 17)]:
    wx = layers_init[idx].W_x.weight
    wg = layers_init[idx].W_g.weight
    print(f"  {name} — W_x std:{wx.std():.6f}  W_g std:{wg.std():.6f}  "
          f"all_zero:{(wx==0).all().item()}")

New module weight init check (should be random, not all-zero):
  DCT-MSA fc[0]     — mean: 0.000197  std: 0.025537  all_zero: False
  AG Gate1 [L13] — W_x std:0.035993  W_g std:0.025539  all_zero:False
  AG Gate2 [L17] — W_x std:0.036263  W_g std:0.036122  all_zero:False


---
## 11. Train YOLO11-S + DCT + AG×2 on FPN Skips

Identical hyperparameters to all previous ablations for fair comparison.

In [ ]:
# dct_ag_model = YOLO("/kaggle/working/yolo11s_dct_ag_init.pt")

# results = dct_ag_model.train(
#     data     = "/kaggle/working/data.yaml",
#     epochs   = 50,
#     imgsz    = 640,
#     batch    = 16,
#     device   = 0,
#     project  = "/kaggle/working/runs",
#     name     = "yolo11s_dct_ag",
#     patience = 10,
#     save     = True,
#     plots    = True,
#     val      = True,
#     workers  = 2,
#     exist_ok = True,
# )

# shutil.make_archive("/kaggle/working/yolo11s_dct_ag_results", "zip",
#                     "/kaggle/working/runs/yolo11s_dct_ag")
# print("Training complete — results zipped ✅")

---
## 12. Training Curves

In [17]:
# RUN_DIR = "/kaggle/working/runs/yolo11s_dct_ag"
BEST_PT = "/kaggle/input/models/rayyansalha/best-50-epochs-dct-and-ag/pytorch/default/1/best.pt"

# df = pd.read_csv(f"{RUN_DIR}/results.csv")
# df.columns = df.columns.str.strip()

# print("── Per-epoch summary ──────────────────────────────────────────")
# print(df[["epoch",
#           "train/box_loss", "val/box_loss",
#           "metrics/mAP50(B)",
#           "metrics/precision(B)",
#           "metrics/recall(B)"]].to_string(index=False))

# fig, axes = plt.subplots(1, 3, figsize=(16, 4))
# fig.suptitle("YOLO11-S + DCT + AG×2 on FPN Skips — Training Curves", fontsize=13)

# axes[0].plot(df["epoch"], df["train/box_loss"], label="train")
# axes[0].plot(df["epoch"], df["val/box_loss"],   label="val")
# axes[0].set_title("Box Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()

# axes[1].plot(df["epoch"], df["train/cls_loss"], label="train")
# axes[1].plot(df["epoch"], df["val/cls_loss"],   label="val")
# axes[1].set_title("Class Loss"); axes[1].set_xlabel("Epoch"); axes[1].legend()

# axes[2].plot(df["epoch"], df["metrics/mAP50(B)"],     label="mAP@0.5")
# axes[2].plot(df["epoch"], df["metrics/precision(B)"], label="Precision")
# axes[2].plot(df["epoch"], df["metrics/recall(B)"],    label="Recall")
# axes[2].set_title("Val Metrics"); axes[2].set_xlabel("Epoch"); axes[2].legend()

# plt.tight_layout()
# plt.savefig("/kaggle/working/yolo11s_dct_ag_curves.png", dpi=120)
# plt.show()

---
## 13. Evaluation

Single evaluation pass on the PyroNear val set — tower camera FP rate and image-level recall.


In [18]:
try:
    del dct_ag_model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f"VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB ✅")

# Load model from Kaggle input (pre-trained, skips training step)
eval_model = YOLO(BEST_PT)

# PyroNear val paths — converted in cell 4 (pyro_yolo)
PYRO_BASE = "/kaggle/working/pyro_yolo"
lbl_val   = Path(f"{PYRO_BASE}/labels/val")
img_val   = Path(f"{PYRO_BASE}/images/val")


def batched_fp_rate(model, hard_neg_imgs, batch_size=200):
    fp = 0
    for i in range(0, len(hard_neg_imgs), batch_size):
        preds = model.predict(source=hard_neg_imgs[i:i+batch_size],
                              imgsz=640, conf=0.25, device=0, verbose=False)
        fp += sum(1 for r in preds if len(r.boxes) > 0)
        torch.cuda.empty_cache()
    return fp, fp / len(hard_neg_imgs) if hard_neg_imgs else 0


def get_hard_negs(img_dir, lbl_dir):
    return [
        str(img_dir / lbl.with_suffix(".jpg").name)
        for lbl in lbl_dir.glob("*.txt")
        if lbl.read_text().strip() == ""
        and (img_dir / lbl.with_suffix(".jpg").name).exists()
    ]

VRAM free: 15.5 GB ✅


In [19]:
print("── EVAL C: PyroNear Val (tower cameras) ───────────────────────")
pyro_val_imgs = sorted(img_val.glob("*.jpg"))  # all val images (pyro-sdis stems)
pyro_val_lbls = [lbl_val / p.with_suffix(".txt").name for p in pyro_val_imgs]
pyro_pos      = sum(1 for l in pyro_val_lbls if l.exists() and l.read_text().strip())
pyro_neg      = sum(1 for l in pyro_val_lbls if l.exists() and not l.read_text().strip())
print(f"  Total images   : {len(pyro_val_imgs)}")
print(f"  Positives      : {pyro_pos}")
print(f"  Hard negatives : {pyro_neg}")

tp_C = fp_C = fn_C = 0
BATCH = 200
for i in range(0, len(pyro_val_imgs), BATCH):
    chunk_imgs = [str(p) for p in pyro_val_imgs[i:i+BATCH]]
    chunk_lbls = pyro_val_lbls[i:i+BATCH]
    preds = eval_model.predict(source=chunk_imgs, imgsz=640,
                               conf=0.25, device=0, verbose=False)
    for pred, lbl_path in zip(preds, chunk_lbls):
        has_gt  = lbl_path.exists() and lbl_path.read_text().strip() != ""
        has_det = len(pred.boxes) > 0
        if     has_gt and     has_det: tp_C += 1
        elif   has_gt and not has_det: fn_C += 1
        elif not has_gt and   has_det: fp_C += 1
    torch.cuda.empty_cache()

fp_rate_C = fp_C / pyro_neg if pyro_neg else 0
recall_C  = tp_C / pyro_pos if pyro_pos else 0
print(f"\n  True Positives    : {tp_C}")
print(f"  False Positives   : {fp_C}")
print(f"  False Negatives   : {fn_C}")
print(f"  FP Rate           : {fp_rate_C:.4f} ({fp_rate_C*100:.1f}%)")
print(f"  Image-level Recall: {recall_C:.4f} ({recall_C*100:.1f}%)")

── EVAL C: PyroNear Val (tower cameras) ───────────────────────
  Total images   : 4099
  Positives      : 3345
  Hard negatives : 754

  True Positives    : 2969
  False Positives   : 435
  False Negatives   : 376
  FP Rate           : 0.5769 (57.7%)
  Image-level Recall: 0.8876 (88.8%)


---
## 14. Results Summary & Save

In [14]:
print("\n" + "="*68)
print(" RESULTS — YOLO11-S + DCT (L10) + AG×2 on FPN Skips")
print("="*68)
print(f"  {'Eval':<38} {'Rec':>8} {'FP Rate':>8}")
print(f"  {'-'*56}")
print(f"  {'PyroNear val (tower cameras)':<38} {recall_C:>8.4f} {fp_rate_C:>8.4f}")
print("="*68)

ablation = [
    ("Baseline",             0.7653, 0.6835, 1.70, 0.7344, 0.8852, 0.6074),
    ("1×CBAM SPPF",          0.7550, 0.6736, 2.19, 0.7308, 0.9046, 0.6008),
    ("3×CBAM Neck",          0.7409, 0.6708, 1.20, 0.7200, 0.8251, 0.4231),
    ("4×CBAM (SPPF+3×Neck)", 0.7574, 0.6873, 2.19, 0.7328, 0.9136, 0.5889),
    ("2×CBAM (SPPF+P4)",     0.7655, 0.6931, 1.85, 0.7306, 0.8957, 0.5915),
    ("AG Skips (#2)",        0.7515, 0.6804, 1.05, 0.7236, 0.8921, 0.5623),
    ("CA+CBAM (#4)",         0.7583, 0.6895, 1.75, 0.7310, 0.9157, 0.6154),
    ("DCT (#3)",             0.7652, 0.6853, 1.75, 0.7333, 0.9133, 0.6167),
]
print(f"\n── Ablation Context ────────────────────────────────────────────────")
print(f"  {'Config':<28} {'mAP-D':>7} {'Rec-D':>7} {'FP-D':>6} {'mAP-C':>7} {'Rec-P':>7} {'FP-P':>6}")
print(f"  {'-'*68}")
for row in ablation:
    name, ma, rc, fp, mc, rp, fpp = row
    print(f"  {name:<28} {ma:>7.4f} {rc:>7.4f} {fp:>5.1f}% {mc:>7.4f} {rp:>7.4f} {fpp*100:>5.1f}%")
print(f"  {'─'*68}")
print(f"  {'DCT + AG (THIS RUN)':<28} {'—':>7} {'—':>7} {'—':>6} {'—':>7} {recall_C:>7.4f} {fp_rate_C*100:>5.1f}%")

dct_ag_results = {
    "model"                : "YOLO11-S + DCT + AG×2 on FPN Skips",
    "train_data"           : "PyroNear (33,636)",
    "recall_pyronear_val"  : round(float(recall_C),  4),
    "fp_rate_pyronear_val" : round(float(fp_rate_C), 4),
    "tp_pyronear"          : int(tp_C),
    "fp_pyronear"          : int(fp_C),
    "fn_pyronear"          : int(fn_C),
}

results_csv = "/kaggle/working/yolo11s_dct_ag_results.csv"
pd.DataFrame([dct_ag_results]).to_csv(results_csv, index=False)
print(f"\nResults saved → {results_csv} ✅")

for k, v in dct_ag_results.items():
    print(f"  {k:<35}: {v}")



 RESULTS — YOLO11-S + DCT (L10) + AG×2 on FPN Skips
  Eval                                        Rec  FP Rate
  --------------------------------------------------------


NameError: name 'recall_C' is not defined

In [ ]:
# Training was skipped — model loaded from Kaggle input dataset.
# Archive the eval results CSV instead.
import shutil
results_csv = "/kaggle/working/yolo11s_dct_ag_results.csv"
if Path(results_csv).exists():
    shutil.copy(results_csv, "/kaggle/working/dct_ag_eval_final.csv")
    print(f"Eval results copied → /kaggle/working/dct_ag_eval_final.csv ✅")
else:
    print("No results CSV found — run eval cells first.")

print(f"Model used: {BEST_PT}")


In [ ]:
import shutil
import os

for d in ["/kaggle/working/pyro_yolo"]:
    if os.path.exists(d):
        shutil.rmtree(d)
        print(f'Deleted {d}')
print('Cleanup complete ✅')
